# Visualise the neighbour-exchange channel

Self-contained. Set the base potential and one gate geometry below, then run all.
The gate parameter reported by `search.py` is `n_delta = 1 - n_target`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from vbc_landscape import swap_landscape, find_swap_endpoints, minimax_barrier

# base potential
R_REP, R_ATT, E_R, E_A = 1.0, 1.5, 20.0, -2.0

# one gate geometry to look at  (r_low, r_high, n_target);  n_delta = 1 - n_target
R_LOW, R_HIGH, N_TARGET = 1.29, 1.50, 0.70

VALENCIES = [2, 4, 6]
r_grid = np.linspace(0.9, 2.2, 401)

In [ ]:
def draw_channel(ax, valency):
    U = swap_landscape(valency, r_grid, R_LOW, R_HIGH, N_TARGET,
                       r_rep=R_REP, r_att=R_ATT, E_r=E_R, E_a=E_A)
    ext = [r_grid[0], r_grid[-1], r_grid[0], r_grid[-1]]
    im = ax.imshow((U / abs(E_A)).T, origin="lower", extent=ext, aspect="equal",
                   vmin=-valency, vmax=0.0, cmap="viridis")
    ax.set_xlabel(r"$r_A$")
    ax.set_ylabel(r"$r_B$")
    title = f"f = {valency}"
    ep = find_swap_endpoints(U, r_grid)
    if ep is not None:
        B = minimax_barrier(U, ep[0], ep[1])
        B_rel = B - max(U[ep[0]], U[ep[1]])
        ax.contour(r_grid, r_grid, U.T, levels=[B], colors="r", linewidths=1.0)
        for (i, j) in ep:
            ax.plot(r_grid[i], r_grid[j], "r*", ms=13)
        title += f"    B/|E_a| = {B_rel / abs(E_A):.3f}"
    ax.set_title(title)
    return im

fig, axes = plt.subplots(1, len(VALENCIES), figsize=(4.2 * len(VALENCIES), 4.0))
axes = np.atleast_1d(axes)
im = None
for ax, v in zip(axes, VALENCIES):
    im = draw_channel(ax, v)
fig.colorbar(im, ax=list(axes), label=r"$U / |E_a|$", shrink=0.85)
plt.show()

## Read back a search result

Point at a `top*_barrier.csv` written by `search.py`.

In [ ]:
import csv, os

path = "./outputs/top100_barrier.csv"
if os.path.exists(path):
    with open(path) as f:
        best = next(csv.DictReader(f))
    print("best gate   r_low=%s  r_high=%s  n_target=%s  (n_delta=%s)"
          % (best["r_low"], best["r_high"], best["n_target"], best["n_delta"]))
    print("worst-case  B/|E_a| = %s    spur/|E_a| = %s"
          % (best["bEa_max"], best["spur_max"]))
    fs = sorted(int(k[5:]) for k in best if k.startswith("bEa_v"))
    bvals = [float(best[f"bEa_v{f}"]) for f in fs]
    fig, ax = plt.subplots(figsize=(4.0, 3.0))
    ax.plot(fs, bvals, "o-")
    ax.set_xlabel("valency f")
    ax.set_ylabel(r"$B / |E_a|$")
    ax.set_title("exchange barrier of the best gate")
    plt.show()
else:
    print("Run search.py first, e.g.  python search.py --out_dir ./outputs")